# Web Break Classifier using extracted features Experiment


Classifies web break events as machine_problem or paper_problem using GPT-4o.
- Batches of 10 events per API call (cheaper + faster)
- Returns structured JSON with selected_class, confidence, and reasoning per event
- Saves results to CSV

In [ ]:
# 1 - Imports & Config

from typing import List
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_community.callbacks import get_openai_callback

import os, re, json, zipfile, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from datetime import datetime

from dotenv import load_dotenv
_ = load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'YOUR_KEY_HERE')
EVENTS_DIR     = r'C:\Users\SREE008\OneDrive - Bertelsmann SE & Co. KGaA\Master Thesis\data\events'
OUT_DIR        = 'webbreak_classifier_output'
os.makedirs(OUT_DIR, exist_ok=True)

LABEL_MAP     = {0: 'machine_problem', 1: 'paper_problem'}
LABEL_MAP_INV = {'machine_problem': 0, 'paper_problem': 1}
POSSIBLE_CLASSES = ['machine_problem', 'paper_problem']
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print('Cell 1 done.')

Cell 1 done.


In [ ]:
# 2 - Data Loading Functions


def read_labels_from_zip(zip_path):
    labels = {}
    with zipfile.ZipFile(zip_path, 'r') as zf:
        lf = [n for n in zf.namelist() if n.lower().endswith('labels.txt')]
        if not lf:
            return labels
        with zf.open(lf[0]) as f:
            lines = f.read().decode('utf-8', errors='ignore').splitlines()
    for line in lines:
        line = line.strip()
        if not line:
            continue
        for sep in [';', ',', '\t', ':', ' ']:
            if sep in line:
                parts = [p.strip() for p in line.split(sep) if p.strip()]
                if len(parts) >= 2 and parts[-1] in ('0', '1'):
                    labels[parts[0].replace('_event.json', '')] = int(parts[-1])
                    break
    return labels


def parse_num(v, d=-1):
    if v is None or v == '':
        return d
    if isinstance(v, (int, float)):
        return v if v != -1 else d
    m = re.search(r'[-+]?\d*\.?\d+', str(v))
    return float(m.group()) if m else d


def dominant_label(scores):
    return max(scores, key=scores.get)


def compress_frames(frames):
    if not frames:
        return 'no frames'
    runs, cur, n = [], dominant_label(frames[0]['scores']), 1
    for fr in frames[1:]:
        lbl = dominant_label(fr['scores'])
        if lbl == cur:
            n += 1
        else:
            runs.append(f'{cur} x{n}')
            cur, n = lbl, 1
    runs.append(f'{cur} x{n}')
    return ' -> '.join(runs)


def event_to_text(event):
    speed = parse_num(event.get('speed'))
    gram  = parse_num(event.get('grammage_weight'))
    plen  = parse_num(event.get('pap_len'))
    meta  = (
        f"Speed: {speed if speed != -1 else 'unknown'} m/s | "
        f"Grammage: {gram if gram != -1 else 'unknown'} g/m2 | "
        f"Paper length: {plen if plen != -1 else 'unknown'} m"
    )
    cams = []
    for vid in event.get('videos', []):
        name   = vid.get('camera_name') or vid.get('camera_id', 'Camera')
        frames = vid.get('frames', [])
        if frames:
            cams.append(f"{name}: {compress_frames(frames)}")
    return meta + '\n' + ('\n'.join(cams) if cams else 'No camera data')


def load_all_events(events_dir):
    zips = sorted(
        [os.path.join(events_dir, f) for f in os.listdir(events_dir)
         if f.lower().endswith('_events.zip')],
        key=lambda p: re.search(r'20\d{4}', os.path.basename(p)).group()
    )
    print(f'Found {len(zips)} zip files')
    texts, labels, ids = [], [], []
    dropped = 0
    for zpath in tqdm(zips, desc='Loading'):
        zip_labels = read_labels_from_zip(zpath)
        if not zip_labels:
            continue
        with zipfile.ZipFile(zpath, 'r') as zf:
            for jname in sorted(n for n in zf.namelist()
                                if n.lower().endswith('_event.json')):
                eid = jname.split('/')[-1].replace('_event.json', '')
                if eid not in zip_labels:
                    dropped += 1
                    continue
                try:
                    with zf.open(jname) as f:
                        ev = json.load(f)
                    texts.append(event_to_text(ev))
                    labels.append(zip_labels[eid])
                    ids.append(eid)
                except Exception:
                    dropped += 1
    labels_arr = np.array(labels, dtype=np.int64)
    print(f'Loaded: {len(texts)} events | dropped: {dropped}')
    print(f'Labels: {np.bincount(labels_arr)}  (0=machine_problem, 1=paper_problem)')
    return texts, labels_arr, ids


print('Cell 2 done.')

Cell 2 done.


In [7]:
%pip install ipywidgets tqdm

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 3 - Load Data + 80/20 Split

texts, labels, ids = load_all_events(EVENTS_DIR)

train_texts, test_texts, train_labels, test_labels, train_ids, test_ids = train_test_split(
    texts, labels, ids,
    test_size=0.20,
    random_state=SEED,
    stratify=labels
)

train_texts  = np.array(train_texts,  dtype=object)
test_texts   = np.array(test_texts,   dtype=object)
train_labels = np.array(train_labels, dtype=np.int64)
test_labels  = np.array(test_labels,  dtype=np.int64)

print(f'Train: {len(train_texts)} events {np.bincount(train_labels)}')
print(f'Test:  {len(test_texts)} events {np.bincount(test_labels)}')
print('\nData ready.')

Found 42 zip files


Loading: 100%|██████████| 42/42 [03:53<00:00,  5.56s/it]

Loaded: 14083 events | dropped: 0
Labels: [11832  2251]  (0=machine_problem, 1=paper_problem)
Train: 11266 events [9465 1801]
Test:  2817 events [2367  450]

Data ready.


In [5]:
# Build few-shot examples from training split (fixed seed, balanced)
N_PER_CLASS = 12
rng = np.random.default_rng(SEED)

train_texts_arr = np.array(train_texts, dtype=object)
train_labels_arr = np.array(train_labels, dtype=np.int64)

idx_machine = np.where(train_labels_arr == 0)[0]
idx_paper   = np.where(train_labels_arr == 1)[0]

pick_machine = rng.choice(idx_machine, size=N_PER_CLASS, replace=False)
pick_paper   = rng.choice(idx_paper,   size=N_PER_CLASS, replace=False)

fewshot_pairs = []
for i in pick_machine:
    fewshot_pairs.append(("machine_problem", train_texts_arr[i]))
for i in pick_paper:
    fewshot_pairs.append(("paper_problem", train_texts_arr[i]))

rng.shuffle(fewshot_pairs)

examples_block = "\n\n".join(
    [f"[Example {k+1} | {lbl}]\n{txt}" for k, (lbl, txt) in enumerate(fewshot_pairs)]
)

In [ ]:

# 4 - Helper Functions

def get_possible_classes():
    """Get the two possible classification labels."""
    return ['machine_problem', 'paper_problem']


def get_few_shot_examples(n_per_class=5, seed=42):
    """
    Pick n_per_class examples per class from TRAINING set.
    Returns list of {event_text, true_label} dicts.
    Never touches test data.
    """
    rng = np.random.default_rng(seed)
    examples = []
    for cls in [0, 1]:
        idx    = np.where(train_labels == cls)[0]
        chosen = rng.choice(idx, size=min(n_per_class, len(idx)), replace=False)
        for i in chosen:
            examples.append({
                'event_text': str(train_texts[i]),
                'true_label': LABEL_MAP[int(train_labels[i])]
            })
    print(f'Few-shot examples: {len(examples)} total '
          f'({sum(1 for e in examples if e["true_label"]=="machine_problem")} machine, '
          f'{sum(1 for e in examples if e["true_label"]=="paper_problem")} paper)')
    return examples


def evaluate(y_true, y_pred, title='Results'):
    print(f'\n{"="*60}')
    print(title)
    print(f'{"="*60}')
    cm = confusion_matrix(y_true, y_pred)
    print(pd.DataFrame(cm,
        index=['true_machine', 'true_paper'],
        columns=['pred_machine', 'pred_paper']).to_string())
    print()
    print(classification_report(y_true, y_pred,
        target_names=['machine_problem', 'paper_problem'],
        digits=3, zero_division=0))


print('Cell 4 done.')

Cell 4 done.


In [ ]:
# 5 - System Prompt + classify_events()
# Prompt built from exact XGBoost decision boundaries (original units)


# Build few-shot examples from training split (fixed seed, balanced)
N_PER_CLASS = 12
rng_fs = np.random.default_rng(SEED)

train_texts_arr  = np.array(train_texts,  dtype=object)
train_labels_arr = np.array(train_labels, dtype=np.int64)

idx_machine = np.where(train_labels_arr == 0)[0]
idx_paper   = np.where(train_labels_arr == 1)[0]

pick_machine = rng_fs.choice(idx_machine, size=N_PER_CLASS, replace=False)
pick_paper   = rng_fs.choice(idx_paper,   size=N_PER_CLASS, replace=False)

fewshot_pairs = []
for i in pick_machine:
    fewshot_pairs.append(("machine_problem", train_texts_arr[i]))
for i in pick_paper:
    fewshot_pairs.append(("paper_problem", train_texts_arr[i]))

rng_fs.shuffle(fewshot_pairs)

examples_block = "\n\n".join(
    [f"[Example {k+1} | {lbl}]\n{txt}" for k, (lbl, txt) in enumerate(fewshot_pairs)]
)

SYSTEM_PROMPT = f"""
You are an expert classifier for paper printing press web break events.
Determine the ROOT CAUSE of each web break: machine_problem or paper_problem.

You will receive JSON with:
- events: list of event text strings (metadata line + per-camera frame sequences)
- possible_classes: ["machine_problem", "paper_problem"]

HOW TO CLASSIFY: FOLLOW THESE STEPS IN ORDER 

STEP 1 - READ THE METADATA LINE FIRST
Every event starts with: Speed: X m/s | Grammage: X g/m2 | Paper length: X m
Extract these three numbers before looking at any camera sequences.

STEP 2 - APPLY THESE DECISION RULES (learned from 14,083 labeled events)

STRONG MACHINE SIGNALS (each alone is meaningful) 
- cam1 tear_ratio < 0.019  : very abrupt break, almost no tear frames : machine
- cam1 rollenwechsel_count > 17  : high mechanical roll-change activity : machine
- cam1 longest_no_defect_run > 0.59  : paper ran cleanly for >59% of sequence : machine
- grammage < 31 g/m²  : very light paper, mechanically stable : machine
- pap_len < 582 m  : very short roll, not enough length to accumulate paper fatigue : machine
- defect_agreement between cameras > 0.994  : both cameras see identical pattern : symmetric = machine

STRONG PAPER SIGNALS (each alone is meaningful)
- grammage > 60 g/m²  : heavy paper, high splice stress : paper
- pap_len > 25000 m  : very long roll, high accumulated fatigue : paper
- speed > 13.8 m/s  : high speed amplifies paper weakness : paper
- cam1 tear_ratio > 0.103  : long messy tear sequence : paper
- cam1 rollenwechsel_count < 5  : almost no roll changes : paper
- defects appearing before position 0.40 in sequence  : paper was weak from start : paper

MEDIUM SIGNALS (use to confirm, not decide alone)
- grammage 31-48 g/m²  : slight machine lean
- grammage 49-60 g/m²  : slight paper lean
- pap_len 582-20000 m  : neutral
- pap_len 20000-25000 m  : slight paper lean
- cam1 defect_acceleration > 0.005  : defects accelerating toward tear : machine
- cam1 defect_acceleration < 0  : defects uniform or decreasing : paper
- cam1 first_defect position > 0.82  : defects very late : machine
- cam1 first_defect position < 0.39  : defects very early : paper
- cam2 first_rollenwechsel position > 0.87  : roll change very late near tear : paper context
- speed < 9.3 m/s  : very slow press : machine lean
- cam1 n_defect_bursts > 248  : extremely fragmented defect pattern : machine

IMPORTANT CORRECTIONS (do NOT use these as signals)
- Kantenfehler count: nearly identical in both classes — ignore as primary signal
- rollenwechsel near tear: appears in both classes — not decisive alone
- tear frames existing: both classes have tears — only the RATIO matters

STEP 3 - COUNT THE SIGNALS
Count how many strong signals point to machine vs paper.
Count how many medium signals support each direction.
The class with more strong signals wins.
If tied on strong signals, use medium signals as tiebreaker.

STEP 4 - SET CONFIDENCE
- high: 2+ strong signals agree on same class, no strong signals contradict
- medium: 1 strong signal, or 2+ medium signals agree
- low: strong signals conflict with each other, or all signals are missing/unknown

STEP 5 - WRITE REASONING
Cite the actual numbers from the event text.
Example: "Grammage=52 g/m² (medium paper lean) + rollenwechsel_count=3 on cam1 (strong paper signal) + defects appear at position 0.35 (strong paper signal) → paper_problem, high confidence"

=== LABELED EXAMPLES (12 machine + 12 paper, shuffled) ===
{examples_block}

OUTPUT FORMAT
{{
  "classifications": [
    {{
      "selected_class": "machine_problem" or "paper_problem",
      "confidence": "high" or "medium" or "low",
      "reasoning": "2-3 sentences citing specific numbers and which step-by-step signals drove the decision."
    }}
  ]
}}

Rules:
- Output valid JSON only, no extra text.
- Order and count must match input events exactly.
- selected_class must be exactly machine_problem or paper_problem.
- high confidence only when 2+ strong signals agree.
"""

def classify_events(event_texts: List[str], possible_classes: List[str]) -> tuple:
    model = ChatOpenAI(
        model='gpt-4o',
        temperature=0.01,
        max_retries=3,
        api_key=OPENAI_API_KEY
    )

    json_parser = JsonOutputParser()

    input_data = {
        'events': event_texts,
        'possible_classes': possible_classes
    }
    in_json = json.dumps(input_data, ensure_ascii=False, indent=2)

    try:
        with get_openai_callback() as cb:
            response = model.invoke([
                SystemMessage(content=SYSTEM_PROMPT),
                HumanMessage(content=f'Input data:\n{in_json}')
            ])

        try:
            parsed = json_parser.parse(response.content)

            if 'classifications' not in parsed:
                raise ValueError("Missing 'classifications' field in response")

            classifications = parsed['classifications']

            if len(classifications) != len(event_texts):
                raise ValueError(
                    f"Number of classifications ({len(classifications)}) "
                    f"doesn't match number of events ({len(event_texts)})"
                )

            selected_classes, confidences, reasonings = [], [], []
            for i, c in enumerate(classifications):
                if 'selected_class' not in c:
                    print(f'Warning: Missing selected_class for event {i}')
                    selected_classes.append(None)
                    confidences.append(None)
                    reasonings.append('Missing selected_class in response')
                else:
                    selected_classes.append(c['selected_class'])
                    confidences.append(c.get('confidence', 'unknown'))
                    reasonings.append(c.get('reasoning', 'No reasoning provided'))

            return cb, selected_classes, confidences, reasonings

        except (json.JSONDecodeError, OutputParserException, ValueError) as parse_error:
            print(f'JSON parsing error: {parse_error}')
            print(f'Raw response: {response.content[:200]}')
            n = len(event_texts)
            return cb, [None]*n, ['low']*n, [f'Parse error: {parse_error}']*n

    except Exception as e:
        print(f'Model invocation error: {e}')
        n = len(event_texts)
        return type('obj', (object,), {'total_cost': 0.0})(),                [None]*n, ['low']*n, [f'Model error: {e}']*n


print('Cell 5 done — XGBoost threshold-informed prompt ready.')
print(f'System prompt length: {len(SYSTEM_PROMPT)} characters')
print(f'Few-shot: {len(fewshot_pairs)} examples ({sum(1 for l,_ in fewshot_pairs if l=="machine_problem")} machine + {sum(1 for l,_ in fewshot_pairs if l=="paper_problem")} paper)')


Cell 5 done — XGBoost threshold-informed prompt ready.
System prompt length: 56351 characters
Few-shot: 24 examples (12 machine + 12 paper)


In [8]:
from langchain_core.messages import SystemMessage, HumanMessage

In [ ]:

# Quick Test (10 events)
# test before full run

possible_classes = get_possible_classes()

# Test with first 10 events from test set
test_sample = test_texts[:10].tolist()

cb_costs, selected_classes, confidences, reasonings = classify_events(
    test_sample, possible_classes
)

print('Classification Results:')
print(f'Total cost: ${cb_costs.total_cost:.6f}')
print(f'Events classified: {len(selected_classes)}')
print()

for i, (text, selected, confidence, reasoning, true_lbl) in enumerate(
        zip(test_sample, selected_classes, confidences, reasonings, test_labels[:10])):
    correct = '✓' if selected == LABEL_MAP[true_lbl] else '✗'
    print(f'{i+1:2d}. {correct} True: {LABEL_MAP[true_lbl]:16s} | Pred: {selected:16s} | Confidence: {confidence}')
    print(f'    Reasoning: {reasoning}')
    print()

Classification Results:
Total cost: $0.077155
Events classified: 10

 1. ✓ True: machine_problem  | Pred: machine_problem  | Confidence: high
    Reasoning: Paper length=17.0 m (strong machine signal) + rollenwechsel_count=4 on cam1 (medium machine lean) → machine_problem, high confidence

 2. ✓ True: machine_problem  | Pred: machine_problem  | Confidence: high
    Reasoning: Grammage=52.0 g/m² (medium machine lean) + numerous rollenwechsel events indicating mechanical activity → machine_problem, high confidence

 3. ✗ True: machine_problem  | Pred: paper_problem    | Confidence: medium
    Reasoning: Speed=11.8 m/s (medium paper lean) + defects appearing early in sequence (medium paper signal) → paper_problem, medium confidence

 4. ✗ True: paper_problem    | Pred: machine_problem  | Confidence: high
    Reasoning: Grammage=40.0 g/m² (strong machine signal) + paper length=19302.0 m (neutral) + no defects until very late in sequence → machine_problem, high confidence

 5. ✓ True: machi

In [ ]:
# 7 - Full Batch Classification
# mirrors Cell 8 from original — batch_classify_products()

possible_classes = get_possible_classes()

# Prepare all test events as list
#all_test_events = test_texts.tolist()
#print(f'Classifying {len(all_test_events)} test events...')

N = 100
rng = np.random.default_rng(SEED)

indices = rng.choice(len(test_texts), size=N, replace=False)

test_texts_arr  = np.array(test_texts, dtype=object)
test_ids_arr    = np.array(test_ids, dtype=object)
test_labels_arr = np.array(test_labels, dtype=np.int64)

all_test_events   = test_texts_arr[indices].tolist()
test_ids_subset   = test_ids_arr[indices]
test_labels_subset = test_labels_arr[indices]

print(f'Classifying {len(all_test_events)} random test events...')

def batch_classify_events(event_texts, possible_classes, batch_size=10):
    """
    Classify events in batches of 10.
    Mirrors batch_classify_products() from original exactly.
    """
    selected_classes  = []
    all_confidences   = []
    all_reasonings    = []
    total_cost        = 0.0
    num_events        = len(event_texts)

    for start in range(0, num_events, batch_size):
        end   = min(start + batch_size, num_events)
        batch = event_texts[start:end]

        cb_costs_batch, selected_batch, confidences_batch, reasonings_batch = \
            classify_events(batch, possible_classes)

        selected_classes.extend(selected_batch)
        all_confidences.extend(confidences_batch)
        all_reasonings.extend(reasonings_batch)
        total_cost += cb_costs_batch.total_cost

        print(f'Classified events {start+1}-{end} (cost: ${cb_costs_batch.total_cost:.6f})')

    class DummyCB:
        def __init__(self, total_cost):
            self.total_cost = total_cost

    return DummyCB(total_cost), selected_classes, all_confidences, all_reasonings


# Run full batch classification
cb_costs_all, selected_classes_all, confidences_all, reasonings_all = \
    batch_classify_events(all_test_events, possible_classes, batch_size=5)

print(f'\nClassification complete. Total cost: ${cb_costs_all.total_cost:.6f}')

Classifying 100 random test events...
Classified events 1-5 (cost: $0.033717)
Classified events 6-10 (cost: $0.039815)
Classified events 11-15 (cost: $0.033402)
Classified events 16-20 (cost: $0.044132)
Classified events 21-25 (cost: $0.040412)
Classified events 26-30 (cost: $0.061420)
Classified events 31-35 (cost: $0.037172)
Classified events 36-40 (cost: $0.035365)
Classified events 41-45 (cost: $0.036132)
Classified events 46-50 (cost: $0.046732)
Classified events 51-55 (cost: $0.038412)
Classified events 56-60 (cost: $0.043810)
Classified events 61-65 (cost: $0.037627)
Classified events 66-70 (cost: $0.043035)
Classified events 71-75 (cost: $0.051115)
Classified events 76-80 (cost: $0.053875)
Classified events 81-85 (cost: $0.082535)
Classified events 86-90 (cost: $0.048262)
Classified events 91-95 (cost: $0.042185)
Classified events 96-100 (cost: $0.042957)

Classification complete. Total cost: $0.892118


In [ ]:
# 8 - Build Results DataFrame + Evaluate (Random 100)


#  Build results table (aligned to the 100 sampled events) 
res_list = []
for i, (event_text, selected, confidence, reasoning, true_lbl, eid) in enumerate(
    zip(
        all_test_events,
        selected_classes_all,
        confidences_all,
        reasonings_all,
        test_labels_subset,
        test_ids_subset
    )
):
    true_name = LABEL_MAP[int(true_lbl)]
    pred_name = selected

    res_list.append({
        "event_id": eid,
        "event_text": event_text,
        "true_label": true_name,
        "predicted_label": pred_name,
        "confidence": confidence,
        "reasoning": reasoning,
        "correct": (pred_name is not None) and (str(pred_name).strip() == true_name)
    })

    # print a single progress line at the end of the 100
    if (i + 1) % 100 == 0:
        print(f"{i+1:4d}. {true_name:16s} => {str(pred_name):16s} ({confidence})")

df_results = pd.DataFrame(res_list)

#Prepare y_true / y_pred (and handle invalid outputs)
y_true = np.array(test_labels_subset, dtype=np.int64)

pred_norm = [(p or "").strip() for p in df_results["predicted_label"].tolist()]
y_pred = np.array([LABEL_MAP_INV.get(p, -1) for p in pred_norm], dtype=np.int64)

invalid_mask = ~np.isin(y_pred, [0, 1])
n_invalid = int(invalid_mask.sum())
print(f"\nInvalid predictions (not in possible classes): {n_invalid}/{len(y_pred)}")

#Evaluate (2x2 confusion matrix on valid predictions only)
valid_mask = ~invalid_mask
y_true_valid = y_true[valid_mask]
y_pred_valid = y_pred[valid_mask]

print("\n" + "=" * 60)
print("Web Break Classifier — GPT-4o Few-Shot (Random 100)")
print("=" * 60)

cm = confusion_matrix(y_true_valid, y_pred_valid, labels=[0, 1])
print(pd.DataFrame(
    cm,
    index=["true_machine", "true_paper"],
    columns=["pred_machine", "pred_paper"]
).to_string())

print()
print(classification_report(
    y_true_valid,
    y_pred_valid,
    target_names=["machine_problem", "paper_problem"],
    digits=3,
    zero_division=0
))

acc_valid = (y_true_valid == y_pred_valid).mean() if len(y_true_valid) else 0.0
acc_all_as_wrong = (y_true == y_pred).mean()  # counts invalids as wrong (since -1 will never equal 0/1)

print(f"Accuracy (valid preds only): {acc_valid:.3f}  on {len(y_true_valid)}/{len(y_true)} events")
print(f"Accuracy (treat invalid as wrong): {acc_all_as_wrong:.3f}  on {len(y_true)}/{len(y_true)} events")

#Confidence breakdown 
print("\nConfidence breakdown:")
print(df_results["confidence"].value_counts(dropna=False).to_string())

#Accuracy by confidence level
print("\nAccuracy by confidence level:")
for conf_level in ["high", "medium", "low"]:
    subset = df_results[df_results["confidence"] == conf_level]
    if len(subset) > 0:
        acc = subset["correct"].mean()
        print(f"  {conf_level:6s}: {acc:.3f} accuracy ({len(subset)} events)")

df_results.head(10)

 100. machine_problem  => machine_problem  (medium)

Invalid predictions (not in possible classes): 0/100

Web Break Classifier — GPT-4o Few-Shot (Random 100)
              pred_machine  pred_paper
true_machine            69          20
true_paper               9           2

                 precision    recall  f1-score   support

machine_problem      0.885     0.775     0.826        89
  paper_problem      0.091     0.182     0.121        11

       accuracy                          0.710       100
      macro avg      0.488     0.479     0.474       100
   weighted avg      0.797     0.710     0.749       100

Accuracy (valid preds only): 0.710  on 100/100 events
Accuracy (treat invalid as wrong): 0.710  on 100/100 events

Confidence breakdown:
confidence
high      66
medium    34

Accuracy by confidence level:
  high  : 0.712 accuracy (66 events)
  medium: 0.706 accuracy (34 events)


,event_id,event_text,true_label,predicted_label,confidence,reasoning,correct
0,FO45_20240205_122405,Speed: 12.2 m/s | Grammage: 38.0 g/m2 | Paper ...,machine_problem,machine_problem,high,Grammage=38.0 g/m² (strong machine signal) + l...,True
1,FO45_20240123_031111,Speed: 8.8 m/s | Grammage: 60.0 g/m2 | Paper l...,machine_problem,paper_problem,medium,Grammage=60.0 g/m² (strong paper signal) + pap...,False
2,FO37_20240820_050937,Speed: 8.7 m/s | Grammage: unknown g/m2 | Pape...,machine_problem,machine_problem,high,Pap_len=60.0 m (strong machine signal) + multi...,True
3,FO51_20240126_141512,Speed: 13.5 m/s | Grammage: 48.0 g/m2 | Paper ...,machine_problem,machine_problem,medium,Grammage=48.0 g/m² (medium machine lean) + pap...,True
4,FO59_20231109_162348,Speed: 15.8 m/s | Grammage: 45.0 g/m2 | Paper ...,machine_problem,machine_problem,high,Pap_len=6.0 m (strong machine signal) + multip...,True
5,FO59_20230309_171350,Speed: unknown m/s | Grammage: unknown g/m2 | ...,machine_problem,machine_problem,high,Paper length=226 m (strong machine signal) + g...,True
6,FO59_20241205_081440,Speed: unknown m/s | Grammage: 40.0 g/m2 | Pap...,machine_problem,machine_problem,medium,Grammage=40 g/m² (slight machine lean) + no de...,True
7,FO46_20240617_183745,Speed: 13.5 m/s | Grammage: 57.0 g/m2 | Paper ...,machine_problem,machine_problem,medium,Grammage=57 g/m² (medium paper lean) + paper l...,True
8,FO53_20240827_163949,Speed: 13.9 m/s | Grammage: 52.0 g/m2 | Paper ...,machine_problem,machine_problem,medium,Grammage=52 g/m² (medium paper lean) + paper l...,True
9,FO45_20240912_213856,Speed: 9.6 m/s | Grammage: 100.0 g/m2 | Paper ...,machine_problem,paper_problem,high,Grammage=100 g/m² (strong paper signal) + pape...,False


In [ ]:

# 9 - Save Results to CSV

os.makedirs(OUT_DIR, exist_ok=True)

out_path = os.path.join(
    OUT_DIR,
    f'webbreak_results_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
)

df_results.to_csv(
    out_path,
    sep=';',
    index=False,
    quotechar='"',
    encoding='utf-8-sig'
)

print(f'Results saved to {out_path}')
print(f'Total events: {len(df_results)}')
print(f'Correct predictions: {df_results["correct"].sum()} ({df_results["correct"].mean():.1%})')
df_results.head()

Results saved to webbreak_classifier_output\webbreak_results_20260307_224423.csv
Total events: 100
Correct predictions: 71 (71.0%)


,event_id,event_text,true_label,predicted_label,confidence,reasoning,correct
0,FO45_20240205_122405,Speed: 12.2 m/s | Grammage: 38.0 g/m2 | Paper ...,machine_problem,machine_problem,high,Grammage=38.0 g/m² (strong machine signal) + l...,True
1,FO45_20240123_031111,Speed: 8.8 m/s | Grammage: 60.0 g/m2 | Paper l...,machine_problem,paper_problem,medium,Grammage=60.0 g/m² (strong paper signal) + pap...,False
2,FO37_20240820_050937,Speed: 8.7 m/s | Grammage: unknown g/m2 | Pape...,machine_problem,machine_problem,high,Pap_len=60.0 m (strong machine signal) + multi...,True
3,FO51_20240126_141512,Speed: 13.5 m/s | Grammage: 48.0 g/m2 | Paper ...,machine_problem,machine_problem,medium,Grammage=48.0 g/m² (medium machine lean) + pap...,True
4,FO59_20231109_162348,Speed: 15.8 m/s | Grammage: 45.0 g/m2 | Paper ...,machine_problem,machine_problem,high,Pap_len=6.0 m (strong machine signal) + multip...,True
